<a href="https://colab.research.google.com/github/taeyoung0524/LoRA-VLM/blob/main/1_VLM_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

sharifa@snu.ac.kr

# 1주차 실습: Toy VLM에서 SmolVLM Fine-tuning까지

## 주차 목표
PyTorch `Tensor`와 `nn.Module`로 Simple Mini VLM의 tensor 흐름을 직접 확인한 뒤, `HuggingFaceTB/SmolVLM-256M-Instruct`를 사용해 단일 이미지 추론, 여러 이미지 zero-shot captioning, COCO validation 평가, COCO subset full fine-tuning까지 한 흐름으로 연결한다.

## Section 구성
- **Section 1:** Simple Mini VLM을 직접 구현해 `image -> vision encoder -> projector -> language decoder -> logits` 흐름과 tensor shape를 확인한다.
- **Section 2:** 이미지 한 장으로 SmolVLM 입력 형식과 `model.generate` 출력을 확인한다.
- **Section 3:** COCO validation 샘플에서 zero-shot caption을 생성하고 BLEU/METEOR/CIDEr-D로 평가한다.
- **Section 4:** COCO subset으로 full fine-tuning을 실행하고 학습 전후 caption을 비교한다.

## 실행 메모
- **실행 환경:** Section 1은 CPU에서도 tensor shape를 확인할 수 있고, Section 2부터 Section 4까지는 GPU 런타임을 기본으로 가정한다. Colab에서는 `런타임 > 런타임 유형 변경 > GPU`를 먼저 선택한다.
- **공통 준비:** 환경 설정과 설치 셀은 앞부분에서 한 번만 실행한다.
- **결과 변수:** 주요 실행 결과는 `result_single`, `result_zs`, `result_ft`로 저장한다.


## Colab 환경 설정

**Colab에서 실행할 때 사용하는 셀이다.** 기본 실행은 Google Drive를 mount한 뒤 프로젝트 경로를 import path에 추가한다. Drive I/O가 느리면 코드 셀 아래쪽의 `setup_colab_workdir(...)` 주석을 해제해 `/content` 로컬 디스크 복사본에서 실행한다.

- **원본 프로젝트 경로:** `/content/drive/MyDrive/VLM`
- **작업 프로젝트 경로:** `/content/VLM`
- **수정 지점:** Drive 폴더명이 다르면 `DRIVE_PROJECT`만 바꾼다.
- **로컬 실행:** Colab이 아니면 안내 메시지만 출력하고 다음 로컬 환경 설정 셀로 넘어간다.


In [ ]:
# Colab 환경 설정
import importlib.util
import sys

DRIVE_PROJECT = '/content/drive/MyDrive/VLM'  # 본인 Drive 폴더명에 맞게 수정

if importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive

    drive.mount('/content/drive')
    if DRIVE_PROJECT not in sys.path:
        sys.path.insert(0, DRIVE_PROJECT)

# 필요할 때만 아래 두 줄의 주석을 해제하세요.
# - Drive 원본을 /content 로컬 디스크로 복사해 I/O 병목을 줄이고 싶을 때
# - 새 Colab 런타임에서 작업 루트와 import path를 한 번에 맞출 때
# from utils.gd_mount import setup_colab_workdir
# setup_colab_workdir(drive_project=DRIVE_PROJECT, mount_drive=False)


## 로컬 환경 설정

**로컬 Jupyter나 VS Code에서 실행할 때 사용하는 셀이다.** 현재 경로의 부모를 탐색해 저장소 루트를 찾는다.

- **탐색 기준:** `index.md`가 있는 디렉터리를 프로젝트 루트로 사용한다.
- **환경 기준:** 로컬에서는 `uv sync` 또는 이미 준비된 `.venv`를 사용한다.
- **설치 방식:** notebook의 `!pip install` 셀보다 저장소의 `uv` 환경을 우선한다.


In [ ]:
# 로컬 환경 설정
import os
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'index.md').exists():
        LOCAL_PROJECT_PATH = candidate
        break
else:
    LOCAL_PROJECT_PATH = Path.cwd()

os.chdir(LOCAL_PROJECT_PATH)
local_project_path = str(LOCAL_PROJECT_PATH)
if local_project_path not in sys.path:
    sys.path.insert(0, local_project_path)
print(f'Local project root: {LOCAL_PROJECT_PATH}')


## GPU 확인

**현재 런타임의 GPU와 PyTorch CUDA 인식 상태를 확인한다.** Colab에서는 환경 설정과 패키지 설치 전후에 한 번 확인하면 된다.


In [ ]:
# GPU 확인
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi 명령을 찾을 수 없습니다. GPU 런타임인지 확인하세요.')

import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 필요 라이브러리 설치

**1주차 전체 실습에 필요한 최소 패키지만 설치한다.** Section 1부터 Section 4까지 실제로 import되는 외부 패키지를 명시적으로 설치한다.

- **설치 기준:** `requirements.txt` 전체 설치가 아니라 1주차 notebook 코드의 import 기준이다.
- **설치 패키지:** `transformers`, `datasets`, `evaluate`, `nltk`, `matplotlib`, `accelerate`, `rich`, `tqdm`, `Pillow`
- **Colab 기본 보존:** `torch`, `torchvision`, `torchaudio`는 기본 환경을 사용한다.
- **로컬 실행:** 로컬에서는 이 셀보다 `uv sync`로 맞춘 저장소 환경을 우선한다.


In [ ]:
# 필요 라이브러리 설치
!pip install -q transformers==4.57.6 datasets==4.8.4 evaluate==0.4.6 nltk==3.9.4 matplotlib==3.10.8 accelerate==1.13.0 rich==13.9.4 tqdm==4.67.3 Pillow==11.3.0
# Colab 기본 설치 패키지는 보존: torch/torchvision/torchaudio


---

# Section 1: Simple Mini VLM 구조 이해

- **무엇을 하는가:** PyTorch `Tensor`와 `nn.Module`만으로 가장 작은 Vision LLM 흐름을 구성하고, image token과 text token이 하나의 sequence로 합쳐지는 과정을 확인한다.
- **핵심 코드:** `MiniVisionEncoder`, `MiniProjector`, `MiniTextDecoder`, `SimpleMiniVLM.forward`.
- **입력:** 무작위 `pixel_values` tensor와 짧은 `input_ids` tensor.
- **출력:** image feature, projected image embedding, concatenated embedding, vocabulary logits의 shape trace.
- **다음 Section 연결:** 같은 구조를 Hugging Face `AutoProcessor`, `AutoModelForImageTextToText`, `model.generate`가 어떻게 감싸는지 확인한다.


## Import

Simple Mini VLM은 작은 vocabulary, 작은 hidden dimension, 작은 image tensor를 사용해 CPU에서도 빠르게 shape를 확인한다.

In [ ]:
# Import

import torch
import torch.nn as nn


## Mini Vision Encoder

`MiniVisionEncoder`는 이미지를 patch 단위로 나누어 visual token sequence를 만든다. 여기서는 `nn.Conv2d(kernel_size=patch_size, stride=patch_size)`를 patch embedding으로 사용한다.


In [ ]:
# Mini Vision Encoder

class MiniVisionEncoder(nn.Module):
    def __init__(self, image_size=224, image_channels=3, vision_dim=32, patch_size=16):
        """
        image_size=224: 입력 이미지의 height/width가 224라고 가정한다.
        image_channels=3: RGB 이미지이므로 channel 수는 3이다.
        vision_dim=32: 각 image patch를 32차원 vector로 표현한다.
        patch_size=16: 이미지를 16×16 patch 단위로 나눈다.
        """
        super().__init__()
        # Vision Transformer에서는 이미지를 patch로 나눈 뒤 각 patch를 vector로 바꾸는데,
        # Conv2d의 kernel_size와 stride를 patch size로 설정하면 patch embedding과 유사하게 동작한다.
        # [TODO 1] Conv2d를 이용해 patch embedding을 수행합니다. 커널 크기와 스트라이드를 패치 크기에 맞게 설정하세요. (hint: 생성자 인자 활용)
        self.patch_embed = nn.Conv2d(
            in_channels=image_channels, # RGB 이미지는 [R, G, B] 세 channel이므로 기본값은 3이다.
            out_channels=vision_dim,    # 각 patch를 몇 차원 vector로 바꿀지 정한다.
            kernel_size=patch_size,           # TODO: 한 번에 볼 영역의 크기를 설정합니다.
            stride=patch_size,                # TODO: 패치가 서로 겹치지 않게 이동할 칸 수를 설정합니다.
        )

        # 입력된 이미지 크기와 패치 크기를 바탕으로 num_patches 자동 계산
        num_patches = (image_size // patch_size) ** 2

        # 1. Positional Embedding 추가
        # Transformer는 token의 순서를 직접 알지 못하기 때문에 위치 정보를 추가해야 한다.
        # 이미지 patch도 sequence로 펼치면 순서 정보가 사라지므로 positional embedding을 더한다.
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, vision_dim))

        # 2. 간단한 Transformer Block 추가
        # patch embedding sequence를 Transformer block에 통과시켜 patch들 사이의 관계를 반영한다.
        # 이 layer는 self-attention과 feed-forward network로 구성된다.
        layer = nn.TransformerEncoderLayer(
            d_model=vision_dim,
            nhead=4,
            batch_first=True, # input: [B, N_patches, D_vision]
            dropout=0.0,
            activation='gelu',
        )
        # encoder layer를 1층짜리 Transformer encoder로 감싼다.
        # nn.TransformerEncoder는 encoder layer를 여러 층 쌓는 모듈이다.
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)

        # 3. Normalization 추가
        # 입력 shape가 [B, N_patches, 32]이면 각 32차원 vector를 정규화한다.
        self.norm = nn.LayerNorm(vision_dim)

    def forward(self, pixel_values):
        # pixel_values: [B, 3, H, W]

        # [TODO 2] patch_embed 레이어에 이미지를 통과시키세요. (hint: 앞서 정의한 self.patch_embed 호출)
        x = self.patch_embed(pixel_values) # TODO

        # x: [B, D_vision, H/patch, W/patch]
        # 결과적으로 입력 [B, 3, 224, 224]는 [B, 32, 14, 14]가 된다.

        # [TODO 3] 추출된 패치 특징맵(x)을 1D 시퀀스로 펼치고(flatten) 차원 순서를 바꾸세요(transpose). (hint: flatten과 transpose 메서드 활용)
        image_features = x.flatten(2).transpose(1, 2) # TODO
        # image_features: [B, N_patches, D_vision]

        # 위치 정보 더하기
        # [TODO 4] flatten된 image_features에 위치 정보(self.pos_embed)를 더하세요. (hint: + 연산자 사용)
        image_features = image_features + self.pos_embed # TODO

        # Transformer & LayerNorm 통과
        image_features = self.transformer(image_features)
        image_features = self.norm(image_features)

        return image_features

# 검증 코드
_test_encoder = MiniVisionEncoder(image_size=224, vision_dim=32, patch_size=16)
_test_input = torch.randn(2, 3, 224, 224)   # pixel_values  : [B, 3, H, W]
try:
    _test_output = _test_encoder(_test_input) # image_features: [B, N_patches, D_vision]
    if _test_output is not None and _test_output.shape == (2, 196, 32):
        print(f"성공! 입력: {tuple(_test_input.shape)} -> 출력: {tuple(_test_output.shape)}")
    else:
        print("실패! 기대되는 입출력 shape는 입력 (2, 3, 224, 224) -> 출력 (2, 196, 32) 입니다.")
except TypeError:
    print("TODO 빈칸을 먼저 채워주세요! (TypeError)")
except AttributeError:
    print("TODO 빈칸이 남아있거나 반환값이 None입니다! (AttributeError)")
except Exception as e:
    print(f"에러 발생: {e}")

## Mini Projector

`MiniProjector`는 vision encoder의 feature dimension을 text decoder의 embedding dimension에 맞춘다. 이 단계가 image feature와 language model 사이의 가장 단순한 multimodal projection layer다.


In [ ]:
# Mini Projector

class MiniProjector(nn.Module):
    def __init__(self, vision_dim=32, text_dim=64):
        """
        vision_dim=32: vision encoder가 출력하는 feature dimension
        text_dim=64: text decoder가 사용하는 embedding dimension
        """
        super().__init__()
        # SmolVLM과 유사하게 단순 Linear layer로 구성
        # Vision encoder의 출력 차원(vision_dim)을 Language model의 입력 차원(text_dim)으로 변환합니다.
        # [TODO 1] 적절한 입력/출력 차원을 설정하세요. (hint: 생성자 인자의 vision_dim, text_dim 활용)
        self.proj = nn.Linear(in_features=vision_dim, out_features=text_dim) # TODO

    def forward(self, image_features):
        # image_features: [B, N_patches, D_vision]

        # 선형 변환을 통해 이미지 피처의 차원을 텍스트 임베딩 차원과 맞춥니다.
        # [TODO 2] 정의한 proj 레이어에 image_features를 통과시키세요. (hint: self.proj 호출)
        image_embeds = self.proj(image_features) # TODO
        # image_embeds: [B, N_patches, D_text]

        return image_embeds

# 검증 코드
_test_projector = MiniProjector(vision_dim=32, text_dim=64)
_test_input = torch.randn(2, 196, 32)
try:
    _test_output = _test_projector(_test_input)
    if _test_output is not None and _test_output.shape == (2, 196, 64):
        print(f"성공! 입력: {tuple(_test_input.shape)} -> 출력: {tuple(_test_output.shape)}")
    else:
        print("실패! 기대되는 입출력 shape는 입력 (2, 196, 32) -> 출력 (2, 196, 64) 입니다.")
except TypeError:
    print("TODO 빈칸을 먼저 채워주세요! (TypeError)")
except AttributeError:
    print("TODO 빈칸이 남아있거나 반환값이 None입니다! (AttributeError)")
except Exception as e:
    print(f"에러 발생: {e}")

- Original Transformer:	ReLU, Position-wise Feed-Forward Network
- Vision Transformer(ViT): GELU, Transformer Encoder 안의 MLP block

## Mini Text Decoder

`MiniTextDecoder`는 text token을 embedding으로 바꾼 뒤, image embedding과 text embedding을 sequence 방향으로 단순 연결한다. 작은 Transformer block과 `lm_head`는 combined sequence의 각 위치를 vocabulary logits로 바꾼다.

여기서는 실제 causal LM decoder를 완전히 재현하지 않고, 구현을 단순화한 decoder-like Transformer block으로 사용한다. 따라서 causal masking보다 image token과 text token이 하나의 sequence로 합쳐지고 vocabulary logits로 이어지는 흐름에 집중한다.


In [ ]:
# Mini Text Decoder

class MiniTextDecoder(nn.Module):
    def __init__(self, vocab_size=1000, text_dim=64):
        super().__init__()
        # 1. 텍스트 토큰 임베딩
        # 입력된 텍스트 토큰 ID를 연속적인 벡터(embedding)로 변환합니다.
        # [TODO 1] 텍스트 토큰을 임베딩하기 위한 레이어를 선언하세요. (hint: 생성자의 vocab_size, text_dim 인자 활용)
        self.token_embed = nn.Embedding(vocab_size, text_dim) # TODO
        # input_ids: [B, L_text] -> text_embeds: [B, L_text, text_dim]

        # 2. 디코더 레이어 정의
        # 텍스트와 이미지 임베딩을 처리할 Transformer Encoder Layer를 정의합니다.
        # 실제로는 Causal LM을 사용하지만, 여기서는 단순화를 위해 EncoderLayer에 마스크를 씌워 사용합니다.
        # gelu를 쓸 수도 있지만 smolvlm의 경우에는 silu activation 함수를 씁니다.
        layer = nn.TransformerEncoderLayer(
            d_model=text_dim,
            nhead=4,
            batch_first=True,
            dropout=0.0,
            activation='gelu',
        )
        self.decoder = nn.TransformerEncoder(layer, num_layers=1)

        # 3. Language Model Head
        # Transformer의 출력을 다시 vocabulary 크기의 로짓(단어별 확률 분포 생성용)으로 변환합니다.
        self.lm_head = nn.Linear(text_dim, vocab_size)
        # hidden_states: [B, L, 64] -> logits:        [B, L, 1000]

    def forward(self, image_embeds, input_ids):
        # image_embeds: [B, N_patches, D_text]
        # input_ids: [B, L_text]

        # 텍스트 토큰을 임베딩 벡터로 변환
        # [TODO 2] input_ids를 텍스트 임베딩으로 변환하세요. (hint: 앞서 정의한 self.token_embed 호출)
        text_embeds = self.token_embed(input_ids) # TODO
        # text_embeds: [B, L_text, D_text]

        # 이미지 임베딩과 텍스트 임베딩을 시퀀스 차원(dim=1)을 기준으로 단순 연결합니다.
        # 결과적으로 [이미지 패치들..., 텍스트 토큰들...] 순서로 합쳐진 하나의 시퀀스가 됩니다.
        # [TODO 3] 이미지 임베딩(image_embeds)과 텍스트 임베딩(text_embeds)을 시퀀스 차원(dim=1)으로 연결하세요. (hint: torch.cat([A, B], dim=1) 형태 사용)
        inputs_embeds = torch.cat([image_embeds, text_embeds], dim=1) # TODO
        # inputs_embeds: [B, N_patches + L_text, D_text]

        seq_len = inputs_embeds.size(1)
        # Causal Mask 생성 (미래의 토큰을 가림)
        # 자기회귀(Autoregressive) 생성을 위해 현재 위치보다 뒤에 있는 미래의 토큰을 보지 못하게 마스킹합니다.
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(inputs_embeds.device)

        # mask와 is_causal을 넘겨 인과적 어텐션 적용
        # 이전 토큰들만 참조하여 현재 토큰의 문맥을 반영한 hidden_states를 계산합니다.
        hidden_states = self.decoder(inputs_embeds, mask=causal_mask, is_causal=True)

        # 다음 단어 예측을 위해 hidden_states를 단어장 크기로 변환합니다.
        logits = self.lm_head(hidden_states)
        # logits: [B, N_patches + L_text, vocab_size]

        return logits, inputs_embeds

# 검증 코드
_test_decoder = MiniTextDecoder(vocab_size=1000, text_dim=64)
_test_img_emb = torch.randn(2, 196, 64)
_test_txt_ids = torch.randint(0, 1000, (2, 8))
try:
    _logits, _embeds = _test_decoder(_test_img_emb, _test_txt_ids)
    if _logits is not None and _embeds is not None and _logits.shape == (2, 204, 1000) and _embeds.shape == (2, 204, 64):
        print(f"성공! 입력: {tuple(_test_img_emb.shape)}, {tuple(_test_txt_ids.shape)} -> 출력 logits: {tuple(_logits.shape)}, embeds: {tuple(_embeds.shape)}")
    else:
        print("실패! 기대되는 입출력 shape는 입력 (2, 196, 64) 및 (2, 8) -> 출력 logits (2, 204, 1000) 및 embeds (2, 204, 64) 입니다.")
except TypeError:
    print("TODO 빈칸을 먼저 채워주세요! (TypeError)")
except AttributeError:
    print("TODO 빈칸이 남아있거나 반환값이 None입니다! (AttributeError)")
except Exception as e:
    print(f"에러 발생: {e}")

## Mini Text Decoder

`MiniTextDecoder`는 text token을 embedding으로 바꾼 뒤, image embedding과 text embedding을 sequence 방향으로 단순 연결한다. 작은 Transformer block과 `lm_head`는 combined sequence의 각 위치를 vocabulary logits로 바꾼다.

여기서는 실제 causal LM decoder를 완전히 재현하지 않고, 구현을 단순화한 decoder-like Transformer block으로 사용한다. 따라서 causal masking보다 image token과 text token이 하나의 sequence로 합쳐지고 vocabulary logits로 이어지는 흐름에 집중한다.

### 부가 설명

`nn.TransformerEncoderLayer`라는 이름 때문에 헷갈리지만, 이 모듈의 내부는 기본적으로 `self-attention -> FFN` 이다.

여기에 causal mask를 넣으면 `masked self-attention -> FFN`이 되고, decoder-only LM의 가장 단순한 형태를 흉내낼 수 있다.

반면 `nn.TransformerDecoderLayer`는 보통 encoder-decoder Transformer용이다. 내부 구조가 대략 이렇다.
```
masked self-attention over tgt
-> cross-attention from tgt to memory
-> FFN
```
즉 TransformerDecoderLayer는 tgt뿐 아니라 memory도 받도록 설계되어 있다.

`decoder_layer(tgt, memory, tgt_mask=...)`

여기서 memory는 보통 encoder 출력이다. 번역 모델로 치면 source sentence encoder output이고, VLM으로 치면 image encoder output을 memory로 줄 수 있다.

그런데 현재 MiniTextDecoder의 목표는 encoder-decoder 구조가 아니다. 현재 toy VLM은 이런 흐름이다.

```
image_embeds + text_embeds를 sequence 방향으로 concat
-> 하나의 causal sequence로 처리
-> lm_head
```
이는 SmolVLM 같은 decoder-only VLM에서 <image> 위치에 image embedding을 넣고 language decoder가 한 sequence로 처리하는 아이디어에 더 가깝다. 그래서 `TransformerDecoderLayer`보다 `TransformerEncoderLayer + causal mask`가 더 단순하다.

`TransformerDecoderLayer`를 바로 쓰면 안 되는 것은 아니다. 다만 구조가 바뀐다.
```
image_embeds = memory
text_embeds = tgt
text가 image memory를 cross-attention으로 봄
```
이렇게 되면 “image token과 text token을 한 sequence로 합쳐 decoder-only LM처럼 처리한다”가 아니라 “image encoder output을 memory로 두고 text decoder가 cross-attention한다”가 된다. 이건 전통적인 encoder-decoder VLM 구조에 더 가깝고, SmolVLM의 입력 결합 방식을 설명하는 데는 덜 직접적이다.

따라서 현재 코드에서 EncoderLayer를 쓰는 이유는:

image_embeds와 text_embeds를 단순 concat한 sequence를 그대로 처리하기 쉽다.
causal mask만 추가하면 decoder-only LM의 masked self-attention 흐름을 보여줄 수 있다.
TransformerDecoderLayer처럼 별도의 memory 입력과 cross-attention 개념을 설명하지 않아도 된다.
실제 SmolVLM도 전통적 encoder-decoder라기보다 image embedding을 language decoder 입력 sequence 안에 넣는 쪽에 가깝다.

## Simple Mini VLM

`SimpleMiniVLM`은 vision encoder, projector, text decoder를 한 번에 연결한다. `forward()`는 image tensor와 text token ids를 받아 logits와 중간 tensor를 함께 반환한다.


In [ ]:
# Simple Mini VLM

class SimpleMiniVLM(nn.Module):
    def __init__(self, image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16):
        super().__init__()
        self.vision_encoder = MiniVisionEncoder(
            image_size=image_size,
            vision_dim=vision_dim,
            patch_size=patch_size,
        )
        self.projector = MiniProjector(
            vision_dim=vision_dim,
            text_dim=text_dim,
        )
        self.text_decoder = MiniTextDecoder(
            vocab_size=vocab_size,
            text_dim=text_dim,
        )

    def forward(self, pixel_values, input_ids):
        # [TODO 1] vision_encoder를 사용해 입력 이미지(pixel_values)에서 시각적 특징(image_features)을 추출하세요.
        image_features = self.vision_encoder(pixel_values) # TODO

        # [TODO 2] projector를 사용해 시각적 특징을 텍스트 임베딩 차원에 맞춘 image_embeds로 변환하세요.
        image_embeds = self.projector(image_features) # TODO

        # [TODO 3] text_decoder에 image_embeds와 텍스트 토큰(input_ids)을 전달하여 예측 결과(logits)와 합쳐진 임베딩(inputs_embeds)을 구하세요.
        # hint: text_decoder는 (logits, inputs_embeds) 두 개의 값을 반환합니다.
        logits, inputs_embeds = self.text_decoder(image_embeds, input_ids) # TODO

        return logits, {
            "image_features": image_features,
            "image_embeds": image_embeds,
            "inputs_embeds": inputs_embeds,
        }

# 검증 코드
_test_vlm = SimpleMiniVLM(image_size=224, vocab_size=1000, vision_dim=32, text_dim=64, patch_size=16)
_test_px = torch.randn(2, 3, 224, 224)
_test_ids = torch.randint(0, 1000, (2, 8))
try:
    _logits, _intermediates = _test_vlm(_test_px, _test_ids)
    if _logits is not None and _logits.shape == (2, 204, 1000):
        print(f"성공! 입력: {tuple(_test_px.shape)}, {tuple(_test_ids.shape)} -> 출력 logits: {tuple(_logits.shape)}")
    else:
        print("실패! 기대되는 입출력 shape는 입력 (2, 3, 224, 224) 및 (2, 8) -> 출력 logits (2, 204, 1000) 입니다.")
except TypeError:
    print("TODO 빈칸을 먼저 채워주세요! (TypeError)")
except AttributeError:
    print("TODO 빈칸이 남아있거나 반환값이 None입니다! (AttributeError)")
except Exception as e:
    print(f"에러 발생: {e}")

## Tensor shape 실행 예시

`224x224` 이미지를 `16x16` patch로 나누면 image token은 `14 * 14 = 196`개다. text token 8개를 붙이면 decoder 입력 sequence 길이는 `196 + 8 = 204`가 된다.

In [ ]:
# Tensor shape 실행 예시

B = 2
H = 224
W = 224
L_TEXT = 8

V = 1000
D_VISION = 32
D_TEXT = 64
PATCH = 16

num_image_tokens = (H // PATCH) * (W // PATCH)
total_sequence_length = num_image_tokens + L_TEXT

assert H % PATCH == 0 and W % PATCH == 0

# 작은 무작위 tensor로 forward 흐름만 확인한다.
torch.manual_seed(0)
pixel_values = torch.randn(B, 3, H, W)
input_ids = torch.randint(low=0, high=V, size=(B, L_TEXT))

simple_mini_vlm = SimpleMiniVLM(
    image_size=H,
    vocab_size=V,
    vision_dim=D_VISION,
    text_dim=D_TEXT,
    patch_size=PATCH,
)
simple_mini_vlm.eval()

intermediates = {}
logits = None
with torch.no_grad():
    logits, intermediates = simple_mini_vlm(pixel_values, input_ids)

shape_trace = {
    "pixel_values": tuple(pixel_values.shape),
    "input_ids": tuple(input_ids.shape),
    "image_features": tuple(intermediates["image_features"].shape),
    "image_embeds": tuple(intermediates["image_embeds"].shape),
    "inputs_embeds": tuple(intermediates["inputs_embeds"].shape),
    "logits": tuple(logits.shape),
}

for name, shape in shape_trace.items():
    print(f"{name:22s}: {shape}")

print(f"{'num_image_tokens':22s}: {num_image_tokens}")
print(f"{'total_sequence_length':22s}: {total_sequence_length}")

result_mini_vlm = {
    "shape_trace": shape_trace,
    "num_image_tokens": num_image_tokens,
    "total_sequence_length": total_sequence_length,
}


## Text Generation

앞서 확인한 `logits`를 이용해 실제로 다음 단어를 예측하고, 이를 다시 입력으로 넣어주는 Autoregressive(자기회귀) 방식의 가장 단순한 Greedy Decoding 루프입니다.

In [ ]:
# Text Generation
def generate_text_compact(model, pixel_values, input_ids, max_new_tokens=5):
    """
    model           SimpleMiniVLM
    pixel_values    이미지 입력
    input_ids       초기 text prompt token ids
    max_new_tokens  새로 생성할 token 개수
    """
    model.eval()
    generated_ids = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # 1. 모델 forward (현재까지의 sequence를 모두 넣음)
            # 이미지와 현재까지 누적된 텍스트 시퀀스를 한 번에 모델에 입력합니다.
            logits, _ = model(pixel_values, generated_ids)

            # 2. 마지막 토큰의 logit에서 가장 확률이 높은 단어(인덱스) 선택 (Greedy Decoding)
            # 시퀀스의 맨 마지막 위치(방금 모델이 예측한 다음 단어)의 로짓 값들 중 최대값의 인덱스를 찾습니다.
            # [TODO 1] 맨 마지막 토큰의 로짓에서 가장 큰 값의 인덱스를 찾으세요. (hint: 마지막 토큰 슬라이싱 [:, -1, :] 후 argmax(dim=-1, keepdim=True) 적용)
            next_token_id = logits[:, -1, :].argmax(dim=-1, keepdim=True) # TODO

            # 3. 예측한 토큰을 기존 sequence 뒤에 이어붙임
            # 이 새로 생성된 토큰을 포함한 전체 시퀀스가 다음 스텝의 입력으로 다시 사용됩니다.
            # [TODO 2] 기존 시퀀스(generated_ids) 뒤에 예측한 토큰(next_token_id)을 이어붙이세요. (hint: torch.cat([A, B], dim=-1) 형태 사용)
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1) # TODO

    return generated_ids

# 데모 셀의 변수를 재사용하여 5개의 새 토큰 생성
try:
    generated_sequence = generate_text_compact(simple_mini_vlm, pixel_values, input_ids, max_new_tokens=5)
    if generated_sequence is not None:
        print(f"초기 입력 sequence shape: {tuple(input_ids.shape)}")
        print(f"생성 후 sequence shape: {tuple(generated_sequence.shape)}")
    else:
        print("실패! TODO 코드를 완성해 주세요.")
except TypeError:
    print("TODO 빈칸을 먼저 채워주세요! (TypeError)")
except AttributeError:
    print("TODO 빈칸이 남아있거나 반환값이 None입니다! (AttributeError)")
except Exception as e:
    print(f"에러 발생: {e}")

## 추가 설명


`MiniVisionEncoder`, `MiniProjector`, `MiniTextDecoder`는 실제 ViT/SmolVLM을 그대로 재현한 모델이 아니라, `image -> visual token -> text embedding space -> logits`로 이어지는 큰 흐름을 보여주기 위한 수업용 toy 구현이다.

- **`MiniVisionEncoder`와 ViT/SigLIP vision encoder:** patch embedding, positional embedding, Transformer encoder, LayerNorm이라는 큰 흐름은 ViT 계열과 비슷하다. 다만 실제 SmolVLM의 vision encoder는 pretrained SigLIP 계열이고, 여러 층의 깊은 encoder, 더 큰 hidden size, 실제 image preprocessing, position 처리, image-text alignment 학습을 포함한다. 여기서는 1층짜리 작은 Transformer로 patch token shape만 확인한다.
- **`MiniProjector`와 SmolVLM connector:** vision feature dimension을 text decoder embedding dimension으로 맞춘다는 역할은 같다. 실제 SmolVLM/Idefics3 계열 connector는 단순 차원 변환뿐 아니라 image token downsampling과 `<image>` token 위치에 visual embedding을 삽입하는 처리를 포함한다. 여기서는 `Linear` 하나로 차원만 바꾸고, image embedding을 text embedding 앞에 단순 연결한다.
- **`MiniTextDecoder`와 실제 language decoder:** text token embedding, image-text sequence 처리, `lm_head`를 통한 vocabulary logits 생성 흐름은 실제 decoder-only LM과 연결된다. 하지만 실제 SmolLM2/Llama-style decoder는 RoPE, RMSNorm, KV cache, attention mask/padding 처리, autoregressive generation loop 등을 포함한다. 여기서는 `nn.TransformerEncoder`에 causal mask를 넣어 decoder-like block처럼 사용한다.

Section 2에서는 Hugging Face `processor`와 `model.generate()`가 image preprocessing, tokenization, image-text 결합, autoregressive generation을 어떻게 감싸는지 실제 SmolVLM으로 확인한다.

---

# Section 2: SmolVLM 첫 실행과 단일 이미지 Captioning

- **무엇을 하는가:** `HuggingFaceTB/SmolVLM-256M-Instruct`를 직접 로드하고, 자유의 여신상 이미지 한 장에 대해 caption을 생성한다.
- **핵심 코드:** `AutoProcessor`, `AutoModelForImageTextToText`, `processor.apply_chat_template`, `model.generate`.
- **입력:** 이미지 URL 1개와 짧은 질문 prompt.
- **출력:** chat template이 적용된 생성 문자열과 이미지-caption 카드.
- **다음 Section 연결:** 한 장에서 확인한 입력 형식을 여러 이미지 zero-shot captioning으로 확장한다.


- 아래 코드는 https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct 에서 가져온 것입니다.
- 모델 로드, 데이터 준비 및 입력만 하면 알아서 결과물이 나옵니다.
- huggingface는 transformer 모델을 아주 쉽게 사용할 수 있는 라이브러리입니다.
- 단, 고도로 추상화되어 있어 직접 코드 내부를 이해하는 것은 어려울 수 있습니다.

## Import와 설치, 환경 설정

notebook 실행에 필요한 외부 패키지와 저장소 공용 모듈을 먼저 불러온다.


In [ ]:
# Import와 설치, 환경 설정

# Section 2부터는 실제 pretrained SmolVLM을 사용하므로 PyTorch와 Hugging Face API를 불러옵니다.
import torch

# AutoProcessor는 이미지 전처리와 텍스트 토큰화를 맡고,
# AutoModelForImageTextToText는 이미지+텍스트 입력으로 텍스트를 생성하는 모델 클래스를 자동 선택합니다.
from transformers import AutoModelForImageTextToText, AutoProcessor

# URL이나 경로에서 이미지를 읽어 PIL Image 형태로 반환하는 Hugging Face helper입니다.
from transformers.image_utils import load_image

# notebook에서 이미지와 생성 caption을 카드 형태로 보여주는 저장소 공용 시각화 함수입니다.
from utils.visualization import show_caption_cards


## 실행 설정과 상수

**핵심:** Section 2는 GPU 런타임에서만 실행한다.

- `DEVICE`: CUDA가 사용 가능할 때만 `cuda`로 고정한다.
- CUDA가 없으면 CPU로 fallback하지 않고 오류를 발생시킨다.
- **다음 단계 연결:** 이 값은 입력 tensor와 모델을 같은 장치로 이동할 때 사용된다.


In [ ]:
# 실행 설정과 상수

# Section 2 이후 실습은 pretrained VLM 추론을 실행하므로 GPU 사용을 전제로 합니다.
# CUDA가 없으면 느리게 CPU fallback하지 않고, 런타임 설정 문제를 바로 알 수 있게 중단합니다.
if not torch.cuda.is_available():
    raise RuntimeError("Section 2는 GPU 런타임에서 실행하세요.")

# 이후 model과 입력 tensor를 같은 GPU 장치로 이동할 때 사용하는 장치 이름입니다.
DEVICE = "cuda"


## 데이터 준비

**핵심:** `load_image`로 원격 이미지 URL을 PIL image로 읽는다.

- **왜 한 장만 쓰는가:** 첫 실행에서는 dataset, dataloader, batch보다 VLM 입력 구조에 집중한다.
- **다음 단계 연결:** 이 PIL image가 processor에서 vision tensor로 변환된다.
- **실습 포인트:** `load_image(...)`에 전달하는 URL 문자열을 바꾸면 같은 모델이 다른 이미지에 어떻게 반응하는지 비교할 수 있다.


In [ ]:
# 데이터 준비

# 단일 이미지 captioning 흐름을 확인하기 위해 공개 URL에서 자유의 여신상 이미지를 불러옵니다.
# load_image는 이미지를 다운로드한 뒤 processor가 받을 수 있는 PIL Image 객체로 변환합니다.
image = load_image("https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg")


## 모델과 Processor 준비

**핵심:** SmolVLM processor와 model을 Hugging Face Hub에서 불러온다.

- `AutoProcessor.from_pretrained(...)`: image placeholder, tokenizer, chat template, vision 전처리를 담당하는 processor를 로드한다.
- `AutoModelForImageTextToText.from_pretrained(...)`: image/text 입력에서 답변 토큰을 생성하는 모델을 로드한다.
- `dtype=torch.bfloat16`: GPU 메모리 사용량을 줄이기 위한 저정밀 dtype이다.
- `_attn_implementation='eager'`: Colab에서 안정적으로 시작하기 위한 attention 구현 선택이다.
- `.to(DEVICE)`: 앞 셀에서 정한 장치로 모델을 이동한다.


In [ ]:
# 모델과 Processor 준비

# 1. Processor 로드
# AutoProcessor는 모델별 image processor와 tokenizer 설정을 Hugging Face Hub에서 함께 불러옵니다.
# 이 processor가 이미지 resize/normalize, chat template 처리, token id 변환을 담당합니다.
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-256M-Instruct")

# 2. Model 로드
# AutoModelForImageTextToText는 model_id의 config를 보고 알맞은 VLM model class를 선택합니다.
# dtype="auto"는 checkpoint가 권장하는 dtype을 사용해 GPU 메모리를 아낍니다.
# _attn_implementation="eager"는 Colab 등에서 호환성이 높은 기본 attention 구현을 사용하게 합니다.
model = AutoModelForImageTextToText.from_pretrained(
    "HuggingFaceTB/SmolVLM-256M-Instruct",
    dtype="auto", # bfloat16           +___********* bfloat16
    _attn_implementation="eager", # Colab 환경 호환성을 위해 eager attention 사용
).to(DEVICE) # model parameter를 입력 tensor와 같은 GPU로 이동합니다.


## 추론 준비

**핵심:** Instruct VLM이 기대하는 chat-format 입력을 만든다.

- `"role": "user"`: 사용자가 질문하는 turn을 나타낸다.
- `{'type': 'image'}`: 이미지가 들어갈 위치를 표시한다.
- `{'type': 'text', 'text': 'Can you describe this image?'}`: 이미지에 대한 자연어 질문이다.
- `add_generation_prompt=True`: assistant가 이어서 답해야 하는 prompt 문자열을 만든다.


In [ ]:
# 추론 준비

# 1. 메시지 형식 정의
# Instruct 모델은 일반 문자열보다 user/assistant role이 있는 대화 형식 입력에 맞춰져 있습니다.
# content 안의 {"type": "image"}는 실제 이미지가 들어갈 위치를 표시합니다.
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"}, # processor가 이 위치를 image placeholder token으로 바꿉니다.
            {"type": "text", "text": "Can you describe this image?"}, # 모델에게 줄 질문 문장입니다.
        ],
    },
]

# 2. Chat template 적용
# apply_chat_template은 messages dict를 SmolVLM tokenizer가 기대하는 하나의 prompt 문자열로 바꿉니다.
# add_generation_prompt=True는 assistant 답변이 시작될 위치를 prompt 끝에 붙입니다.
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

# 3. 모델 입력 텐서 생성
# processor는 prompt를 input_ids로, image를 pixel_values 등 vision 입력 tensor로 변환합니다.
# return_tensors="pt"를 지정해 PyTorch Tensor 형태로 받습니다.
inputs = processor(text=prompt, images=[image], return_tensors="pt")

# 4. 장치 이동
# model은 GPU에 있으므로 processor가 만든 모든 입력 tensor도 같은 GPU로 이동해야 합니다.
inputs = inputs.to(DEVICE)


## 추론 실행

**핵심:** `model.generate`로 답변 토큰을 생성하고 decode한다.

- `processor.batch_decode(generated_ids, skip_special_tokens=True)`는 입력 prompt와 생성 답변이 포함된 sequence를 문자열로 바꾼다.
- generated ids에서 입력 prompt를 출력에서 제외하려면 `batch_decode` 첫 번째 인자를 주석 처리된 slicing 줄로 바꾼다.
- `skip_special_tokens=True`: 남아 있는 special token을 제거한다.


In [ ]:
# 추론 실행

# 1. 텍스트 생성
# **inputs는 input_ids, pixel_values 같은 항목을 model.generate의 keyword argument로 풀어 전달합니다.
# max_new_tokens는 prompt 뒤에 새로 생성할 token 수의 상한입니다.
# 내부적으로는 이전 token들을 보고 다음 token을 하나씩 생성하는 autoregressive decoding이 실행됩니다.
generated_ids = model.generate(**inputs, max_new_tokens=500)

# 2. 결과 디코딩
# generated_ids는 정수 token id이므로 processor.batch_decode로 사람이 읽는 문자열로 되돌립니다.
# 현재 코드는 prompt와 생성 답변이 함께 포함된 전체 sequence를 decode합니다.
generated_texts = processor.batch_decode(
    # generated_ids,
    # 입력 prompt를 출력에서 제외하고 생성 답변만 보고 싶다면 아래 줄처럼 generated_ids를 slicing하면 됩니다.
    generated_ids[:, inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True, # <eos> 같은 tokenizer 특수 token은 최종 문자열에서 제거합니다.
)


## 결과 확인 및 시각화

**핵심:** decode된 생성 문자열을 확인하고, 같은 셀에서 이미지-caption 카드로 시각화한다.

- `generated_texts[0]`: 현재 decode 설정에 따라 prompt와 답변이 함께 보일 수 있는 생성 문자열이다.
- `result_single['samples']`: 이미지, 입력 prompt, 파일명을 담는 표시용 샘플.
- `result_single['captions']`: 시각화 helper가 `PRED`로 보여 줄 모델 출력.
- **다음 Section 연결:** 같은 결과 구조를 여러 이미지 captioning 결과에도 재사용한다.


In [ ]:
# 결과 확인 및 시각화

# samples는 이미지와 원래 prompt 정보를, captions는 모델이 생성한 문자열을 담습니다.
result_single = {
    "samples": [
        {
            "image": image,
            "caption": "Can you describe this image?",
            "filename": "statue-of-liberty.jpg",
        }
    ],
    "captions": [generated_texts[0]],
}

# 이미지, prompt, 생성 caption을 notebook 안에서 카드 형태로 표시합니다.
show_caption_cards(
    result_single["samples"],
    predicted_captions=result_single["captions"],
    cols=1,
    caption_label="PROMPT",
    title="Week 1 Period 2: Single-image Captioning",
)


전체 코드입니다.
```python
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load images
image = load_image("https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg")

# Initialize processor and model
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-256M-Instruct")
model = AutoModelForVision2Seq.from_pretrained(
    "HuggingFaceTB/SmolVLM-256M-Instruct",
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2" if DEVICE == "cuda" else "eager",
).to(DEVICE)

# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Can you describe this image?"}
        ]
    },
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])
"""
Assistant: The image depicts a large, historic statue of liberty, located in New York City. The statue is a green, cylindrical structure with a human figure at the top, holding a torch. The statue is situated on a pedestal that resembles the statue of liberty, which is located on a small island in the middle of a body of water. The water surrounding the island is calm, reflecting the blue sky and the statue.
In the background, there are several tall buildings, including the Empire State Building, which is visible in the distance. These buildings are made of glass and steel, and they are positioned in a grid-like pattern, giving them a modern look. The sky is clear, with a few clouds visible, indicating fair weather.
The statue is surrounded by trees, which are green and appear to be healthy. There are also some small structures, possibly houses or buildings, visible in the distance. The overall scene suggests a peaceful and serene environment, typical of a cityscape.
The image is taken during the daytime, likely during the day of the statue's installation. The lighting is bright, casting a strong shadow on the statue and the water, which enhances the visibility of the statue and the surrounding environment.
To summarize, the image captures a significant historical statue of liberty, situated on a small island in the middle of a body of water, surrounded by trees and buildings. The sky is clear, with a few clouds visible, indicating fair weather. The statue is green and cylindrical, with a human figure holding a torch, and is surrounded by trees, indicating a peaceful and well-maintained environment. The overall scene is one of tranquility and historical significance.
"""
```

# Huggingface 전체 코드

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image

def log_var_info(name, var):
    shape_info = getattr(var, 'shape', None)
    if shape_info is None and hasattr(var, '__len__'):
        shape_info = f"len={len(var)}"
    print(f"{name} type: {type(var)}, shape/info: {shape_info}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load images
image = load_image("https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg")
log_var_info("image", image)

# Initialize processor and model
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-256M-Instruct")
log_var_info("processor", processor)

model = AutoModelForImageTextToText.from_pretrained(
    "HuggingFaceTB/SmolVLM-256M-Instruct",
    torch_dtype=torch.bfloat16,
    _attn_implementation="eager",
).to(DEVICE)
log_var_info("model", model)

# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Can you describe this image?"}
        ]
    },
]
log_var_info("messages", messages)

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
log_var_info("prompt", prompt)

inputs = processor(text=prompt, images=[image], return_tensors="pt")
log_var_info("inputs", inputs)

# print로 inputs의 각 tensor shape 및 type 출력
for k, v in inputs.items():
    print(f"inputs['{k}'] type: {type(v)}, shape : {v.shape}")

inputs = inputs.to(DEVICE)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)
log_var_info("generated_ids", generated_ids)

generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)
log_var_info("generated_texts", generated_texts)

print(generated_texts[0])

In [ ]:
# 모델의 전체 파라미터와 학습 가능한(trainable) 파라미터 수를 계산하여 출력합니다.

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Trainable Ratio: {100 * trainable_params / total_params:.4f}%")

---

# Section 3: COCO Validation Zero-shot 평가

- **무엇을 하는가:** COCO validation 샘플을 가져와 zero-shot caption을 생성하고 reference caption과 metric으로 비교한다.
- **핵심 코드:** `ZeroShotConfig`, `sample_coco_image_subset`, `create_coco_dataloader`, `generate_captions_for_batch`, `compute_caption_metrics`.
- **입력:** COCO validation 이미지/정답 caption subset.
- **출력:** 생성 caption, BLEU/METEOR/CIDEr-D metric, caption card와 metric chart.
- **다음 Section 연결:** 같은 COCO captioning 흐름을 full fine-tuning 전후 비교의 기준선으로 확장한다.


## COCO 데이터셋 설명

COCO Captions는 MS COCO 이미지에 사람이 작성한 영어 장면 설명을 연결한 image captioning 데이터셋이다.
각 이미지는 보통 5개 이상의 reference caption을 가지므로, 같은 장면을 여러 자연어 표현으로 학습·평가할 수 있다.

자세한 데이터셋 설명은 [docs/coco_captions_dataset.md](docs/coco_captions_dataset.md)를 참고한다.

예시 데이터:
- image: `COCO_train2014_000000000009.jpg`
- captions:
  - `A meal is presented in brightly colored plastic trays.`
  - `A bunch of trays that have different food.`
  - `Colorful dishes holding meat, vegetables, fruit, and bread.`

데이터 수:
- `train2014`: 약 82K images, 약 414K captions
- `val2014`: 약 40K images, 약 202K captions
- 이미지당 caption: 보통 5개 이상


## Import와 설치, 환경 설정

notebook 실행에 필요한 외부 패키지와 저장소 공용 모듈을 먼저 불러온다.


In [ ]:
# Import와 설치, 환경 설정

# dataclass는 zero-shot 실행 설정을 하나의 config 객체로 묶기 위해 사용합니다.
from dataclasses import dataclass

# perf_counter는 batch caption 생성 시간을 측정해 이미지당 latency를 계산할 때 사용합니다.
from time import perf_counter

# Any는 sample dict, processor, model처럼 외부 library 객체의 type hint에 사용합니다.
from typing import Any

# caption metric 계산, COCO dataloader, device/logging/model helper, 시각화 helper를 불러옵니다.
from utils.caption_metrics import compute_caption_metrics
from utils.coco_dataloader import create_coco_dataloader, create_coco_dataset, sample_coco_image_subset
from utils.device_utils import get_device_info, resolve_device
from utils.logger_utils import get_logger
from utils.smolvlm_utils import generate_captions_for_batch, load_model_and_processor
from utils.visualization import show_caption_metric_comparison


## 실행 설정

**핵심:** COCO validation zero-shot 평가에 필요한 설정을 정의한다.

- `model_id`: 평가에 사용할 SmolVLM checkpoint.
- `prompt`: COCO 이미지에 적용할 caption prompt.
- `metrics`: BLEU, METEOR, CIDEr-D 계산 목록.
- `ZeroShotConfig`: split, sample 수, batch size, device, generation 길이를 묶는 설정 객체.


In [ ]:
# 실행 설정

# Section 3 전용 logger입니다. 진행 상황과 metric을 notebook 출력에 남깁니다.
ZERO_SHOT_LOGGER = get_logger(__name__)

# zero-shot 평가에서 바꿀 가능성이 있는 값들을 dataclass 하나로 모읍니다.
# slots=True는 오타로 새 attribute가 생기는 일을 막고 객체를 가볍게 만듭니다.
@dataclass(slots=True)
class ZeroShotConfig:
    split: str = "validation" # COCO validation split에서 이미지를 가져옵니다.
    num_images: int = 4 # 빠른 실습을 위해 평가 이미지 수를 작게 둡니다.
    batch_size: int = 2 # 한 번에 generate할 이미지 수입니다.
    seed: int = 42 # subset sampling을 재현 가능하게 만드는 seed입니다.
    max_new_tokens: int = 64 # caption 생성 길이 상한입니다.
    device: str | None = "cuda" # 기본 실행 장치는 GPU입니다.
    model_id: str = "HuggingFaceTB/SmolVLM-256M-Instruct"
    prompt: str = "Describe this image in one concise English sentence."
    metrics: tuple[str, ...] = ("bleu", "meteor", "cider_d") # 계산할 caption metric 목록입니다.


## 샘플별 예측 로그 출력

**핵심:** COCO zero-shot 평가에서 샘플별 reference, prediction, latency를 로그로 남기는 helper를 정의한다.

- `_log_sample_prediction`: sample filename, ground-truth caption, generated caption을 같은 순서로 출력한다.
- latency가 계산된 경우 이미지당 평균 생성 시간도 함께 기록한다.
- **다음 단계 연결:** caption 생성 셀의 batch loop 안에서 이 함수를 호출한다.


In [ ]:
# 샘플별 예측 로그 출력

# 각 이미지의 파일명, 정답 caption, 예측 caption, latency를 같은 형식으로 출력하는 작은 helper입니다.
# 여러 셀에서 재사용하지는 않지만, batch loop 안의 로깅 코드를 읽기 쉽게 분리합니다.
def _log_sample_prediction(
    *,
    index: int,
    total: int,
    sample: dict[str, Any],
    caption: str,
    latency_sec: float | None,
) -> None:
    # 현재 몇 번째 sample을 처리했는지와 COCO 파일명을 먼저 출력합니다.
    ZERO_SHOT_LOGGER.info("[%s/%s] %s", index, total, sample["filename"])

    # GT는 ground truth caption, PRED는 모델이 생성한 caption입니다.
    ZERO_SHOT_LOGGER.info("  GT: %s", sample["caption"])
    ZERO_SHOT_LOGGER.info("  PRED: %s", caption)

    # batch caption 수가 0인 특수 상황이 아니면 이미지당 평균 latency도 출력합니다.
    if latency_sec is not None:
        ZERO_SHOT_LOGGER.info("  Latency: %.2fs", latency_sec)


## COCO subset 생성

**핵심:** COCO validation split에서 평가에 사용할 이미지 단위 subset을 고른다.

- `ZeroShotConfig`: split, sample 수, batch size, metric 목록을 묶는다.
- `sample_coco_image_subset`: 중복 이미지를 피한 COCO image sample 목록을 만든다.
- **재현성:** `seed`, `split`, `num_images`를 설정 객체에서 가져와 같은 subset을 다시 만들 수 있게 한다.


In [ ]:
# COCO subset 생성

# config 객체를 만들고 현재 실행 설정을 로그로 남겨 재현성을 확보합니다.
config_zs = ZeroShotConfig()
ZERO_SHOT_LOGGER.info(
    "Run config: model=%s split=%s num_images=%s batch_size=%s device=%s",
    config_zs.model_id,
    config_zs.split,
    config_zs.num_images,
    config_zs.batch_size,
    config_zs.device,
)

# COCO validation split에서 image 단위 sample을 지정한 개수만큼 뽑습니다.
# streaming=True는 전체 dataset을 먼저 다운로드하지 않고 순차적으로 읽게 합니다.
# shuffle=True와 seed는 매 실행에서 같은 subset 순서를 얻기 위해 함께 사용합니다.
ZERO_SHOT_LOGGER.info("Loading COCO samples")
samples = sample_coco_image_subset(
    max_images=config_zs.num_images,
    split=config_zs.split,
    streaming=True,
    shuffle=True,
    show_progress=False,
    seed=config_zs.seed,
)
ZERO_SHOT_LOGGER.info("Loaded %s COCO samples", len(samples))


## Dataloader 등 데이터 준비

**핵심:** 선택된 COCO sample을 dataset과 dataloader로 바꿔 batch 추론 단위를 만든다.

- `create_coco_dataset`: image, caption, filename을 담은 dataset 객체를 만든다.
- `create_coco_dataloader`: `batch_size`에 맞춰 순서대로 batch를 만든다.
- **다음 단계 연결:** caption 생성 셀은 `dataloader`를 순회하며 batch별 이미지를 모델에 넣는다.


In [ ]:
# Dataloader 등 데이터 준비

# list 형태의 COCO sample을 PyTorch Dataset처럼 접근할 수 있는 객체로 감쌉니다.
ZERO_SHOT_LOGGER.info("Creating dataset and dataloader")
dataset = create_coco_dataset(samples)

# DataLoader는 batch_size 단위로 sample을 묶어 caption 생성 loop에 공급합니다.
# 이미 sample_coco_image_subset에서 seed 기반 shuffle을 했으므로 여기서는 순서를 유지합니다.
dataloader = create_coco_dataloader(
    dataset,
    batch_size=config_zs.batch_size,
    shuffle=False,
)


## 모델 로드

**핵심:** COCO zero-shot 평가용 설정으로 같은 SmolVLM model/processor를 불러온다.

- 기본 device는 `cuda`이며, CUDA가 없으면 `resolve_device`에서 오류가 발생한다.
- `torch_dtype="auto"`로 GPU에 맞는 dtype 선택을 공용 helper에 맡긴다.


In [ ]:
# 모델 로드

# config의 device 문자열을 실제 torch.device로 확정하고, 로그/저장용 device 정보도 함께 얻습니다.
device = resolve_device(config_zs.device)
device_info = get_device_info(config_zs.device)
ZERO_SHOT_LOGGER.info("Resolved device: %s", device_info["device"])

# SmolVLM processor와 model을 한 번에 로드하는 저장소 공용 helper를 사용합니다.
# torch_dtype="auto"는 checkpoint에 맞는 dtype 선택을 library에 맡깁니다.
# prefer_flash_attention=False는 호환성을 위해 eager attention 경로를 사용하게 합니다.
ZERO_SHOT_LOGGER.info("Loading model and processor")
model_bundle = load_model_and_processor(
    model_id=config_zs.model_id,
    device=device,
    torch_dtype="auto",
    prefer_flash_attention=False,
)

# 이후 셀에서 자주 쓰는 processor/model만 별도 변수로 꺼내 둡니다.
processor = model_bundle["processor"]
model = model_bundle["model"]
ZERO_SHOT_LOGGER.info("Attention implementation: %s", model_bundle["attn_implementation"])


## Caption 생성

**핵심:** dataloader batch를 순회하면서 COCO 이미지별 caption을 생성한다.

- `generate_captions_for_batch`: batch image 목록을 한 번에 모델에 넣는다.
- `_log_sample_prediction`: reference와 prediction을 같은 sample 순서로 기록한다.
- `predictions`: metric 계산과 시각화에 사용할 생성 caption 목록이다.


In [ ]:
# Caption 생성

# 모든 batch에서 생성한 caption을 순서대로 누적할 list입니다.
predictions: list[str] = []
ZERO_SHOT_LOGGER.info("Generating captions")

# 1. 배치 순회
# dataloader에서 config_zs.batch_size만큼 이미지와 sample metadata를 가져옵니다.
for batch in dataloader:
    # batch 시작 시간을 기록해 caption 생성 latency를 계산합니다.
    batch_start_time = perf_counter()

    # 2. 배치 단위 모델 추론
    # 여러 장의 이미지를 한 번에 모델에 입력하여 효율적으로 caption을 생성합니다.
    # helper 내부에서 chat template 구성, processor 호출, model.generate, decode가 실행됩니다.
    try:
        batch_captions = generate_captions_for_batch(
            model=model,
            processor=processor,
            # [TODO 1] 현재 배치의 PIL 이미지 리스트를 전달하세요. (hint: batch 딕셔너리의 "images" 키 값을 list()로 감싸기)
            images=None, # TODO
            device=device,
            prompt=config_zs.prompt,      # 모든 이미지에 같은 captioning prompt를 사용합니다.
            max_new_tokens=config_zs.max_new_tokens,
            # [TODO 2] Greedy Decoding 방식을 사용하기 위해 샘플링(do_sample)을 비활성화(False)하세요.
            do_sample=None,              # TODO
        )
    except Exception as e:
        ZERO_SHOT_LOGGER.error("TODO 코드를 완성해주세요! %s", e)
        break

    # 3. 레이턴시 계산 및 로깅 준비
    # batch 전체 시간을 caption 수로 나눠 이미지당 평균 생성 시간을 추정합니다.
    batch_latency_sec = perf_counter() - batch_start_time
    latency_per_image = batch_latency_sec / len(batch_captions) if batch_captions else None
    start_index = len(predictions)
    batch_samples = list(batch["samples"])

    # 4. sample별 결과 누적
    # strict=True는 sample 수와 caption 수가 다르면 즉시 오류를 내어 결과 불일치를 잡습니다.
    for offset, (sample, caption) in enumerate(zip(batch_samples, batch_captions, strict=True), start=1):
        _log_sample_prediction(
            index=start_index + offset,
            total=len(samples),
            sample=sample,
            caption=caption,
            latency_sec=latency_per_image,
        )
        predictions.append(caption)


## Metric 계산

**핵심:** 생성 caption을 COCO reference caption과 비교해 정량 평가값을 계산한다.

- `references`: sample의 ground-truth caption을 문자열로 정리한 목록이다.
- `compute_caption_metrics`: BLEU, METEOR, CIDEr-D 등 설정된 metric을 계산한다.
- **해석 포인트:** 사람이 보기에는 괜찮은 caption도 n-gram 기반 metric에서는 낮게 나올 수 있다.


In [ ]:
# Metric 계산

# compute_caption_metrics는 이미지마다 여러 reference caption을 받을 수 있으므로
# 각 sample의 captions list를 [reference1, reference2, ...] 형태로 정리합니다.
references = [
    [str(caption).strip() for caption in sample.get("captions", [sample["caption"]])]
    for sample in samples
]

# 모델 예측 caption과 COCO reference caption을 비교해 BLEU/METEOR/CIDEr-D를 계산합니다.
ZERO_SHOT_LOGGER.info("Computing metrics")
metrics = compute_caption_metrics(predictions, references, config_zs.metrics)
ZERO_SHOT_LOGGER.info("Completed zero-shot captioning for %s samples", len(samples))
ZERO_SHOT_LOGGER.info("Metrics: %s", metrics)


## 결과 정리 및 시각화

**핵심:** COCO zero-shot caption과 metric을 함께 확인한다.


In [ ]:
# 결과 정리 및 시각화

# Section 3의 주요 산출물을 하나의 dict로 모아 이후 셀이나 강의 설명에서 재사용합니다.
result_zs = {
    "device_info": device_info,
    "samples": samples,
    "predictions": predictions,
    "metrics": metrics,
}

# 각 이미지의 reference caption과 모델 예측 caption을 카드 형태로 확인합니다.
show_caption_cards(
    result_zs["samples"],
    predicted_captions=result_zs["predictions"],
    title="SmolVLM Zero-shot Captioning",
)

# 계산된 caption metric을 막대그래프로 요약합니다.
_ = show_caption_metric_comparison(
    {"Zero-shot": result_zs["metrics"]},
    title="Zero-shot caption metrics",
)


---

# Section 4: COCO Subset Full Fine-tuning

- **무엇을 하는가:** COCO subset으로 SmolVLM을 full fine-tuning하고, fine-tuning 전후 caption을 비교한다.
- **핵심 코드:** `FinetuneConfig`, `sample_coco_image_subset`, `Trainer`, `generate_captions_for_batch`, `compute_caption_metrics`.
- **입력:** COCO train 기반 학습 subset, COCO validation 기반 검증/테스트 subset, 이미지별 대표 caption.
- **출력:** checkpoint, train/eval metric JSON, zero-shot vs fine-tuned caption 비교, before/after figure.
- **실행 주의:** 가장 오래 걸리는 Section이므로 GPU 런타임을 기본으로 사용하고, 이미 저장된 checkpoint가 있으면 재사용한다.


## Import와 설치, 환경 설정

notebook 실행에 필요한 외부 패키지와 저장소 공용 모듈을 먼저 불러온다.


In [ ]:
# Import와 설치, 환경 설정

# inspect는 현재 설치된 transformers.Trainer가 어떤 인자를 받는지 확인하는 데 사용합니다.
import inspect

# json은 checkpoint 설정 파일을 읽어 현재 config와 일치하는지 비교할 때 사용합니다.
import json

# transformers 전체 module은 TrainingArguments, Trainer, TrainerCallback을 함께 참조하기 위해 불러옵니다.
import transformers

# asdict는 dataclass config를 JSON으로 저장 가능한 dict로 변환합니다.
from dataclasses import asdict

# tqdm은 fine-tuning 전후 caption 생성 진행률을 notebook에 표시합니다.
from tqdm.auto import tqdm

# GPU 메모리 정리, SmolVLM 로드/저장, SFT collator, 학습 인자, 시각화 helper를 불러옵니다.
from utils.device_utils import release_cuda_memory
from utils.smolvlm_utils import (
    align_model_generation_config_with_tokenizer,
    build_sft_collate_fn,
    load_pretrained_model,
    resolve_auto_model_class,
    resolve_torch_dtype,
    save_caption_comparison_figure,
)
from utils.training_utils import build_training_arguments_kwargs, count_parameters, save_json
from utils.visualization import show_caption_comparison_cards


## 실행 설정

**핵심:** COCO full fine-tuning에 필요한 데이터 split, 데이터 크기, 학습 하이퍼파라미터, 저장 경로를 정의한다.

- `model_name`: fine-tuning을 시작할 base SmolVLM checkpoint.
- `prompt`: fine-tuning 전후 caption 비교에 사용할 prompt.
- `seed`: COCO sampling과 Trainer 내부 shuffle 재현성에 사용한다.
- `train_split`: train subset을 가져올 원본 COCO split이다.
- `eval_split`: validation/test subset을 가져올 원본 COCO split이다.
- `device`: 기본값은 `cuda`이며, CUDA가 없으면 실행을 중단한다.
- `FinetuneConfig`: train/val/test image 수, batch size, gradient accumulation, epoch, learning rate, output 경로를 묶는다.


In [ ]:
# 실행 설정

# Section 4 전용 logger입니다. 데이터 준비, 학습, 평가, 저장 과정을 같은 형식으로 출력합니다.
FINETUNE_LOGGER = get_logger(__name__)

# full fine-tuning 실습에서 사용하는 모든 주요 하이퍼파라미터와 경로를 한곳에 모읍니다.
@dataclass(slots=True)
class FinetuneConfig:
    model_name: str = "HuggingFaceTB/SmolVLM-256M-Instruct" # fine-tuning할 base model입니다.
    prompt: str = "Describe this image in one concise English sentence." # 학습/추론에 공통으로 쓰는 instruction입니다.
    train_images: int = 1000 # train split에서 사용할 unique image 수입니다.
    val_images: int = 50 # validation metric 계산에 사용할 image 수입니다.
    test_images: int = 150 # before/after caption metric 비교에 사용할 image 수입니다.
    compare_images: int = 4 # 카드 시각화에 보여줄 validation image 수입니다.
    seed: int = 42 # COCO subset sampling 재현성을 위한 seed입니다.
    train_split: str = "train" # 학습 이미지를 가져올 원본 COCO split입니다.
    eval_split: str = "validation" # validation/test 이미지를 가져올 원본 COCO split입니다.
    device: str | None = "cuda" # full fine-tuning은 GPU 실행을 기본으로 합니다.
    per_device_train_batch_size: int = 2 # GPU 한 장에서 한 step에 처리할 train sample 수입니다.
    per_device_eval_batch_size: int = 4 # 평가/생성에서 한 번에 처리할 sample 수입니다.
    gradient_accumulation_steps: int = 8 # batch size * gradient_accumulation_steps = 16으로 유지
    num_train_epochs: int = 1 # 실습 시간 안에 끝나도록 epoch 수를 작게 둡니다.
    learning_rate: float = 2e-5 # full fine-tuning용 learning rate입니다.
    weight_decay: float = 0.01 # optimizer의 L2 regularization 계수입니다.
    warmup_ratio: float = 0.03 # 초반 learning rate warmup 비율입니다.
    logging_steps: int = 5 # 이 step 간격마다 training loss를 로그로 출력합니다.
    max_new_tokens: int = 48 # before/after caption 생성 길이 상한입니다.
    data_dir: Path = Path("data/1-4-coco-full-finetuning") # split manifest 등 데이터 산출물 경로입니다.
    output_dir: Path = Path("model/1-4-coco-full-finetuning") # checkpoint와 report 저장 경로입니다.


## Device 및 환경 설정

**핵심:** full fine-tuning 실행 설정, 저장 디렉터리, GPU device 정보를 준비한다.

- `FinetuneConfig`: 학습/평가 이미지 수, batch size, epoch, 저장 경로를 묶는다.
- `resolve_device`: `cuda` device를 확인하고 사용할 GPU를 정한다.
- `device_info.json`: 실행 환경을 결과 디렉터리에 저장해 재현 정보를 남긴다.


In [ ]:
# Device 및 환경 설정

# config 객체를 만들고 train/val/test 전체 이미지 수를 계산합니다.
config_ft = FinetuneConfig()
total_images = config_ft.train_images + config_ft.val_images + config_ft.test_images

# 실행 시작 시 핵심 설정을 로그로 남겨 notebook output만 봐도 어떤 run인지 알 수 있게 합니다.
FINETUNE_LOGGER.info(
    "Starting week 1 fine-tuning run | model=%s train_split=%s eval_split=%s total_images=%s train=%s val=%s test=%s epochs=%s output_dir=%s",
    config_ft.model_name,
    config_ft.train_split,
    config_ft.eval_split,
    total_images,
    config_ft.train_images,
    config_ft.val_images,
    config_ft.test_images,
    config_ft.num_train_epochs,
    config_ft.output_dir,
)

# 데이터 manifest와 model checkpoint/report를 저장할 폴더를 미리 생성합니다.
config_ft.data_dir.mkdir(parents=True, exist_ok=True)
config_ft.output_dir.mkdir(parents=True, exist_ok=True)

# 문자열 설정값을 실제 torch.device로 바꾸고, CUDA 사용 가능 여부 같은 환경 정보를 수집합니다.
device = resolve_device(config_ft.device)
device_info = get_device_info(config_ft.device)
FINETUNE_LOGGER.info(
    "Using device=%s cuda_available=%s cuda_device_count=%s",
    device_info["device"],
    device_info["cuda_available"],
    device_info["cuda_device_count"],
)

# device_info 안의 device 객체는 JSON 직렬화가 어려울 수 있으므로 문자열로 변환해 저장합니다.
device_info_payload = {
    **device_info,
    "device": str(device_info["device"]),
}
save_json(config_ft.output_dir / "device_info.json", device_info_payload)
FINETUNE_LOGGER.info("Saved device info to %s", config_ft.output_dir / "device_info.json")


## 데이터 준비

**핵심:** `config_ft.train_split`에서 학습 subset을, `config_ft.eval_split`에서 validation/test subset을 가져와 이미지별 대표 caption 하나만 사용해 Trainer용 dataset을 준비한다.

- `sample_coco_image_subset`: train source와 validation/test source를 각각 이미지 단위 subset으로 만든다.
- `create_coco_dataset`: 이미지 sample의 대표 `caption` 하나를 학습 row로 사용한다.
- `comparison_samples`: fine-tuning 전후 caption을 눈으로 비교할 validation 이미지다.
- split manifest 저장은 다음 셀에서 별도로 처리한다.


In [ ]:
# 데이터 준비

# train split에서는 학습에 사용할 unique image sample을 가져옵니다.
# sample_coco_image_subset은 한 이미지당 하나의 대표 caption을 sample["caption"]에 담습니다.
FINETUNE_LOGGER.info("Loading COCO train subset and grouping unique images")
train_image_samples = sample_coco_image_subset(
    max_images=config_ft.train_images,
    split=config_ft.train_split,
    streaming=True,
    shuffle=True,
    show_progress=True,
    seed=config_ft.seed,
)

# validation split에서는 validation용 이미지와 test용 이미지를 한 번에 뽑은 뒤 아래에서 나눕니다.
# 같은 seed를 사용하므로 같은 설정으로 다시 실행하면 같은 순서의 sample이 나옵니다.
FINETUNE_LOGGER.info("Loading COCO validation subset for validation/test images")
eval_image_samples = sample_coco_image_subset(
    max_images=config_ft.val_images + config_ft.test_images,
    split=config_ft.eval_split,
    streaming=True,
    shuffle=True,
    show_progress=True,
    seed=config_ft.seed,
)

# 앞쪽 val_images개는 Trainer evaluate용 validation set으로 사용합니다.
val_end = config_ft.val_images
val_image_samples = eval_image_samples[:val_end]

# 나머지는 fine-tuning 전후 metric을 비교하는 held-out test set으로 사용합니다.
test_image_samples = eval_image_samples[val_end:]

# validation sample 중 일부만 카드 시각화에 사용해 notebook 출력이 너무 길어지지 않게 합니다.
comparison_samples = val_image_samples[: config_ft.compare_images]

# sample list를 Hugging Face Trainer가 사용할 수 있는 Dataset 형태로 감쌉니다.
train_dataset = create_coco_dataset(train_image_samples)
val_dataset = create_coco_dataset(val_image_samples)

# 실제 준비된 이미지 수와 dataset row 수를 로그로 확인합니다.
FINETUNE_LOGGER.info(
    "Prepared dataset split | train_images=%s val_images=%s test_images=%s comparison_images=%s",
    len(train_image_samples),
    len(val_image_samples),
    len(test_image_samples),
    len(comparison_samples),
)
FINETUNE_LOGGER.info(
    "Dataset rows | train_rows=%s val_rows=%s",
    len(train_dataset),
    len(val_dataset),
)


## Split manifest 저장

**핵심:** 재현 가능한 비교를 위해 원본 COCO split, seed, 목표 split 크기, 실제 cocoid 목록을 manifest로 저장한다.

- `train_source_split`: train subset을 가져온 원본 COCO split이다.
- `eval_source_split`: validation/test subset을 가져온 원본 COCO split이다.
- `target_counts`: train/validation/test/comparison에 의도한 이미지 수다.
- `*_cocoids`: 실제로 각 split에 배정된 COCO image id 목록이다.


In [ ]:
# Split manifest 저장

# 어떤 COCO split과 seed로 어떤 image id를 뽑았는지 기록해 재현 가능하게 만듭니다.
split_manifest = {
    "train_source_split": config_ft.train_split,
    "eval_source_split": config_ft.eval_split,
    "seed": config_ft.seed,
    "target_counts": {
        "train_images": config_ft.train_images,
        "val_images": config_ft.val_images,
        "test_images": config_ft.test_images,
        "compare_images": config_ft.compare_images,
    },
    # cocoid list를 저장하면 나중에 같은 subset이 맞는지 빠르게 비교할 수 있습니다.
    "train_cocoids": [int(sample["cocoid"]) for sample in train_image_samples],
    "val_cocoids": [int(sample["cocoid"]) for sample in val_image_samples],
    "test_cocoids": [int(sample["cocoid"]) for sample in test_image_samples],
    "comparison_cocoids": [int(sample["cocoid"]) for sample in comparison_samples],
}

# manifest는 data_dir에 저장하고, checkpoint 재사용 판단에는 별도의 run_config를 사용합니다.
save_json(config_ft.data_dir / "split_manifest.json", split_manifest)
FINETUNE_LOGGER.info("Saved split manifest to %s", config_ft.data_dir / "split_manifest.json")


## 모델 경로 준비 및 모델 로드

**핵심:** full fine-tuning에 필요한 checkpoint 경로와 dtype, 파라미터 규모를 함께 기록한다.

- `checkpoint_dir`: 학습된 full fine-tuning checkpoint를 저장하거나 재사용할 위치다.
- `resolve_torch_dtype`: generation보다 메모리 사용이 큰 학습 단계에 맞춰 dtype을 고른다.
- `count_parameters`: full fine-tuning에서 업데이트되는 파라미터 규모를 확인한다.


In [ ]:
# 모델 경로 준비 및 모델 로드

# Trainer가 저장하거나 재사용할 checkpoint 폴더입니다.
checkpoint_dir = config_ft.output_dir / "trainer"
FINETUNE_LOGGER.info("Loading processor and model")

# GPU와 checkpoint에 맞는 dtype을 결정한 뒤 SmolVLM processor/model을 로드합니다.
torch_dtype = resolve_torch_dtype(device=device, torch_dtype="auto")
model_bundle = load_model_and_processor(
    model_id=config_ft.model_name,
    device=device,
    torch_dtype=torch_dtype,
    prefer_flash_attention=False,
)

# bundle에서 실제 학습에 사용할 객체와 dtype 정보를 꺼냅니다.
processor = model_bundle["processor"]
model = model_bundle["model"]
model_torch_dtype = model_bundle["torch_dtype"]
FINETUNE_LOGGER.info("Model loaded on %s", device)

# full fine-tuning에서는 전체 parameter가 trainable인지 확인하는 것이 중요합니다.
total_params = count_parameters(model)
trainable_params = count_parameters(model, trainable_only=True)
FINETUNE_LOGGER.info(
    "Model parameter summary | total_params=%s trainable_params=%s trainable_ratio=%.4f%%",
    total_params,
    trainable_params,
    100 * (trainable_params / total_params if total_params else 0.0),
)


## 공통 caption 생성 함수

**핵심:** comparison/test split을 batch 단위로 captioning하는 공통 helper를 정의한다.

- `generate_split_captions`: split별 image sample을 batch로 나누고 `generate_captions_for_batch`를 호출해 caption 목록을 반환한다.
- zero-shot baseline과 fine-tuned 평가에서 같은 prompt, batch size, generation 설정을 재사용한다.


In [ ]:
# 공통 caption 생성 함수

# fine-tuning 전 zero-shot model과 fine-tuning 후 model에 같은 방식으로 caption을 생성하기 위한 helper입니다.
def generate_split_captions(
    *,
    label: str,
    model: Any,
    processor: Any,
    sample_splits: dict[str, list[dict[str, Any]]],
    device: Any,
    config: FinetuneConfig,
) -> dict[str, list[str]]:
    # caption 생성에서는 dropout 등을 끄기 위해 evaluation mode로 전환합니다.
    model.eval()

    # output_key별 caption list를 담습니다. 예: comparison, test
    outputs: dict[str, list[str]] = {}

    # sample_splits에 들어 있는 각 split을 같은 로직으로 순회합니다.
    for output_key, image_samples in sample_splits.items():
        FINETUNE_LOGGER.info("Running %s caption generation for %s samples", label.lower(), output_key)
        captions: list[str] = []

        # per_device_eval_batch_size 단위로 index를 움직이며 진행률을 표시합니다.
        progress = tqdm(
            range(0, len(image_samples), config.per_device_eval_batch_size),
            desc=f"{label} {output_key} captions",
        )
        for start_index in progress:
            # 현재 batch에 해당하는 sample slice를 만듭니다.
            batch_samples = image_samples[start_index : start_index + config.per_device_eval_batch_size]

            # 공용 caption helper가 processor/generate/decode를 처리하고 caption list를 반환합니다.
            captions.extend(
                generate_captions_for_batch(
                    model=model,
                    processor=processor,
                    images=[sample["image"] for sample in batch_samples],
                    device=device,
                    prompt=config.prompt,
                    max_new_tokens=config.max_new_tokens,
                    do_sample=False,
                )
            )

        # 현재 split의 caption을 결과 dict에 저장합니다.
        outputs[output_key] = captions
    return outputs


## Zero-shot baseline

**핵심:** fine-tuning 전 comparison/test split caption을 먼저 생성한다.

- `zero_shot_outputs['comparison']`: 눈으로 비교할 validation 이미지 caption이다.
- `zero_shot_outputs['test']`: metric 계산에 사용할 test 이미지 caption이다.


In [ ]:
# Zero-shot baseline

# fine-tuning 전 모델의 caption 품질을 저장해 두기 위해 비교용 split을 준비합니다.
# comparison은 카드 시각화용, test는 metric 계산용입니다.
caption_splits = {
    "comparison": comparison_samples,
    "test": test_image_samples,
}

# 아직 학습하지 않은 base SmolVLM으로 caption을 생성합니다.
# 이후 같은 sample_splits에 대해 fine-tuned model caption을 생성해 before/after를 비교합니다.
zero_shot_outputs = generate_split_captions(
    label="Zero-shot",
    model=model,
    processor=processor,
    sample_splits=caption_splits,
    device=device,
    config=config_ft,
)


## Checkpoint 확인

**핵심:** 기존 checkpoint가 현재 fine-tuning 설정과 일치할 때만 학습을 건너뛰고 재사용한다.

- `checkpoint_file_exists`: 저장된 단일 또는 sharded 모델 파일 존재 여부를 확인한다.
- `checkpoint_config_matches`: checkpoint와 함께 저장한 설정 metadata가 현재 `FinetuneConfig`와 같은지 확인한다.
- 설정이 다르면 base model에서 다시 학습하고 같은 checkpoint directory를 덮어쓴다.


In [ ]:
# Checkpoint 확인

# Trainer를 구성하기 전에 남아 있는 CUDA cache를 정리해 OOM 가능성을 줄입니다.
FINETUNE_LOGGER.info("Releasing CUDA cache before Trainer setup")
release_cuda_memory(device)

# 현재 실행 설정을 checkpoint metadata로 저장/비교하기 위해 JSON 가능한 dict로 만듭니다.
# sft_label_mask는 collator가 assistant 답변 부분만 loss로 학습하도록 만든 현재 학습 형식을 나타냅니다.
config_payload = {
    **asdict(config_ft),
    "data_dir": str(config_ft.data_dir),
    "output_dir": str(config_ft.output_dir),
    "sft_label_mask": "assistant_only_v1",
}
checkpoint_meta_path = checkpoint_dir / "run_config.json"

# checkpoint 폴더에 실제 model weight 파일이 있는지 확인합니다.
checkpoint_weight_exists = any(checkpoint_dir.glob("*.safetensors")) or any(
    checkpoint_dir.glob("pytorch_model*.bin")
)
checkpoint_file_exists = (
    checkpoint_dir.is_dir()
    and (checkpoint_dir / "config.json").is_file()
    and (checkpoint_weight_exists or (checkpoint_dir / "model.safetensors.index.json").is_file())
)

# 저장된 run_config가 현재 config와 같아야만 checkpoint를 재사용합니다.
checkpoint_config_matches = (
    checkpoint_meta_path.is_file()
    and json.loads(checkpoint_meta_path.read_text(encoding="utf-8")).get("config") == config_payload
)

# train/eval 결과 변수는 checkpoint 재사용 여부에 따라 나중에 값이 채워집니다.
train_result_metrics: dict[str, Any] | None = None
eval_metrics: dict[str, Any] | None = None
log_history: list[dict[str, Any]] = []
used_existing_checkpoint = checkpoint_file_exists and checkpoint_config_matches

# weight는 있지만 설정이 다르면 이전 run 산출물이므로 base model에서 다시 학습합니다.
if checkpoint_file_exists and not checkpoint_config_matches:
    FINETUNE_LOGGER.info("Existing checkpoint config differs from the current run; training from base model")

# 설정까지 일치하는 checkpoint가 있으면 학습을 건너뛰고 저장된 fine-tuned model을 로드합니다.
if used_existing_checkpoint:
    FINETUNE_LOGGER.info("Found matching trainer checkpoint at %s; reusing it", checkpoint_dir)
    del model
    release_cuda_memory(device)
    auto_model_class = resolve_auto_model_class(transformers)
    model = load_pretrained_model(
        auto_model_class,
        str(checkpoint_dir),
        torch_dtype=model_torch_dtype,
    )
    model = model.to(device)

    # checkpoint와 tokenizer의 special token 설정을 맞춰 generation 경고와 불일치를 줄입니다.
    align_model_generation_config_with_tokenizer(model, processor.tokenizer)

# gradient checkpointing/training 중에는 cache 사용이 충돌하거나 메모리를 더 쓸 수 있어 꺼 둡니다.
if getattr(device, "type", None) == "cuda" and getattr(model.config, "use_cache", None) is not None:
    model.config.use_cache = False


## Trainer 구성

**핵심:** 학습 실행 여부와 관계없이 `Trainer`를 구성한다.

- 새 학습이 필요한 경우 같은 `Trainer`로 `train()`을 실행한다.
- 기존 checkpoint를 재사용하는 경우에도 같은 `Trainer`로 `evaluate()`를 실행한다.
- 기존 checkpoint와 현재 설정이 다르면 `overwrite_output_dir=True`로 새 결과를 저장한다.
- `StepLossLogger`는 notebook progress table과 별개로 main process에서만 `logging_steps`마다 training loss를 로그로 출력한다.


In [ ]:
# Trainer 구성

# Trainer가 logging_steps마다 만든 loss log를 저장소 logger 형식으로 다시 출력하는 callback입니다.
class StepLossLogger(transformers.TrainerCallback):
    # transformers.Trainer는 on_log 같은 callback hook을 통해 학습 중간 이벤트에 개입할 수 있습니다.
    def on_log(self, args, state, control, logs=None, **kwargs):
        # 분산 학습에서는 main process만 출력해야 같은 로그가 GPU 수만큼 반복되지 않습니다.
        # loss가 없는 evaluation log 등은 여기서 처리하지 않습니다.
        if not state.is_world_process_zero or not logs or "loss" not in logs:
            return

        # 현재 global step, epoch, loss, learning rate를 사람이 읽기 쉬운 한 줄 로그로 출력합니다.
        FINETUNE_LOGGER.info(
            "Training loss | step=%s epoch=%.2f loss=%.4f lr=%.2e",
            state.global_step,
            (logs.get("epoch") or 0.0),
            logs["loss"],
            (logs.get("learning_rate") or 0.0),
        )

# 1. Training Arguments 설정
# build_training_arguments_kwargs는 transformers 버전 차이를 흡수하면서 TrainingArguments 인자를 구성합니다.
# 여기에는 batch size, epoch, learning rate, logging_steps, eval/save strategy 등이 들어갑니다.
training_args_kwargs = build_training_arguments_kwargs(
    training_arguments_cls=transformers.TrainingArguments,
    output_dir=str(checkpoint_dir),
    config=config_ft,
    device=device,
    torch_dtype=model_torch_dtype,
)

# 기존 checkpoint가 현재 설정과 다르면 같은 output_dir을 새 학습 결과로 덮어쓸 수 있게 합니다.
if checkpoint_file_exists and not checkpoint_config_matches:
    training_args_kwargs["overwrite_output_dir"] = True

# 실제 Trainer가 사용할 TrainingArguments 객체를 만듭니다.
training_args = transformers.TrainingArguments(**training_args_kwargs)

# 2. Data Collator 설정
# collate_fn은 image/caption sample을 model input tensor와 labels로 변환합니다.
# image_token_id를 전달해 processor가 만든 <image> token 위치와 image embedding을 맞출 수 있게 합니다.
collate_fn = build_sft_collate_fn(
    processor,
    image_token_id=getattr(model.config, "image_token_id", None),
    prompt=config_ft.prompt,
)

# 3. Trainer 객체 초기화
# Trainer는 model, dataset, collator, optimizer/scheduler, logging/eval/save loop를 묶어 관리합니다.
trainer_kwargs: dict[str, Any] = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "data_collator": collate_fn,
    "callbacks": [StepLossLogger()], # 우리가 만든 loss logger callback을 추가합니다.
}

# transformers 버전에 따라 processor를 넘기는 인자명이 processing_class 또는 tokenizer로 다릅니다.
# inspect로 현재 설치 버전의 Trainer signature를 보고 맞는 인자만 넣습니다.
trainer_signature = inspect.signature(transformers.Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = processor
else:
    trainer_kwargs["tokenizer"] = processor.tokenizer

# 위에서 준비한 인자들로 Trainer를 최종 생성합니다.
trainer = transformers.Trainer(**trainer_kwargs)


## 학습

**핵심:** 새 checkpoint가 필요할 때만 Trainer fine-tuning을 실행한다.

- 기존 checkpoint를 재사용하는 경우 학습만 건너뛰고, 평가는 다음 셀에서 그대로 실행한다.
- 새로 학습하는 경우 `train_result_metrics`에 Trainer가 반환한 학습 metric을 저장하고 checkpoint를 바로 저장한다.
- **시간 주의:** 이 셀은 GPU 환경에서 가장 오래 걸리는 단계다.


In [ ]:
# 학습

# 현재 설정과 일치하는 checkpoint가 있으면 긴 학습을 반복하지 않고 재사용합니다.
if used_existing_checkpoint:
    FINETUNE_LOGGER.info("Trainer fine-tuning skipped because an existing checkpoint is being reused")
else:
    # checkpoint가 없거나 설정이 달라졌으면 base model에서 full fine-tuning을 실행합니다.
    FINETUNE_LOGGER.info("Starting Trainer fine-tuning")
    train_result = trainer.train()

    # Trainer가 반환한 train loss, runtime 같은 metric을 나중에 저장하기 위해 보관합니다.
    train_result_metrics = train_result.metrics
    FINETUNE_LOGGER.info("Training complete")

    # 학습이 끝난 model과 processor/tokenizer를 checkpoint_dir에 저장합니다.
    trainer.save_model()
    save_pretrained = getattr(processor, "save_pretrained", None)
    if callable(save_pretrained):
        save_pretrained(checkpoint_dir)

    # 현재 config도 함께 저장해 다음 실행 때 checkpoint 재사용 여부를 판단합니다.
    save_json(checkpoint_meta_path, {"config": config_payload})
    FINETUNE_LOGGER.info("Saved trainer checkpoint to %s", checkpoint_dir)


## 추론 및 평가 준비

**핵심:** `Trainer` validation 평가를 실행한 뒤 caption generation에 필요한 cache 설정과 GPU cache 상태를 정리한다.

- `trainer.evaluate()`: validation dataset 기준 loss 등 평가 metric을 계산한다.
- `use_cache=True`: caption generation 단계에서는 decoder cache를 다시 사용할 수 있게 한다.
- `release_cuda_memory`: Trainer 평가 뒤 남은 CUDA cache를 비운다.


In [ ]:
# 추론 및 평가 준비

# validation dataset으로 Trainer 평가를 실행해 eval_loss 등 학습 후 평가 metric을 얻습니다.
FINETUNE_LOGGER.info("Starting Trainer evaluation")
eval_metrics = trainer.evaluate()

# Trainer 내부 log history에는 step별 loss, eval result 등이 들어 있어 report로 저장합니다.
log_history = list(trainer.state.log_history)

# 학습 중 꺼 두었던 cache를 caption generation 전에는 다시 켜서 추론을 빠르게 합니다.
if getattr(device, "type", None) == "cuda" and getattr(model.config, "use_cache", None) is not None:
    model.config.use_cache = True

# 평가 직후 남은 CUDA cache를 정리한 뒤 fine-tuned caption 생성을 시작합니다.
FINETUNE_LOGGER.info("Releasing CUDA cache before fine-tuned caption generation")
release_cuda_memory(device)


## 평가

**핵심:** fine-tuned caption을 생성하고 comparison/test split별 예측을 나눈다.

- fine-tuned caption 생성은 baseline과 같은 `generate_split_captions` helper를 재사용한다.
- metric 비교는 다음 `Before / After metric 계산` 셀에서 별도로 수행한다.


In [ ]:
# 평가

# fine-tuning된 model로 comparison/test split caption을 다시 생성합니다.
fine_tuned_outputs = generate_split_captions(
    label="Fine-tuned",
    model=model,
    processor=processor,
    sample_splits=caption_splits,
    device=device,
    config=config_ft,
)

# zero-shot 결과에서 카드 시각화용 comparison caption과 metric 계산용 test caption을 꺼냅니다.
zero_shot_predictions = zero_shot_outputs["comparison"]
zero_shot_metric_predictions = zero_shot_outputs["test"]

# fine-tuned 결과에서도 같은 key를 사용해 comparison/test caption을 꺼냅니다.
fine_tuned_predictions = fine_tuned_outputs["comparison"]
fine_tuned_metric_predictions = fine_tuned_outputs["test"]


## Before / After metric 계산

**핵심:** test split에서 zero-shot과 fine-tuned caption을 같은 reference 기준으로 비교한다.

- `test_references`: COCO reference caption 목록이다.
- `comparison_metrics['delta']`: fine-tuned metric에서 zero-shot metric을 뺀 변화량이다.


In [ ]:
# Before / After metric 계산

# test sample마다 reference caption list를 만듭니다.
# fine-tuning 전후 모두 같은 reference와 비교해야 metric 차이가 공정합니다.
test_references = [[str(sample["caption"])] for sample in test_image_samples]

# zero-shot caption과 fine-tuned caption을 같은 metric 함수로 평가합니다.
zero_shot_metrics = compute_caption_metrics(zero_shot_metric_predictions, test_references)
fine_tuned_metrics = compute_caption_metrics(fine_tuned_metric_predictions, test_references)

# metric별 before, after, delta를 한 dict에 모아 저장/시각화에서 재사용합니다.
comparison_metrics = {
    "zero_shot": zero_shot_metrics,
    "fine_tuned": fine_tuned_metrics,
    "delta": {
        metric_name: round(fine_tuned_metrics[metric_name] - zero_shot_metrics[metric_name], 6)
        for metric_name in zero_shot_metrics
    },
}

# 주요 metric 변화를 한 줄 로그로 요약합니다.
FINETUNE_LOGGER.info(
    "Comparison metrics | bleu=%.4f->%.4f meteor=%.4f->%.4f cider_d=%.4f->%.4f",
    zero_shot_metrics["bleu"],
    fine_tuned_metrics["bleu"],
    zero_shot_metrics["meteor"],
    fine_tuned_metrics["meteor"],
    zero_shot_metrics["cider_d"],
    fine_tuned_metrics["cider_d"],
)


## Report

**핵심:** before/after 비교용 JSON 구조를 만들고 terminal log로 요약을 출력한다.

- `before_after_report['samples']`: 비교 이미지별 ground truth, zero-shot, fine-tuned caption을 담는다.
- `before_after_report['metrics']`: zero-shot과 fine-tuned metric, delta를 담는다.
- terminal log는 metric summary와 sample별 caption을 순서대로 보여준다.


In [ ]:
# Report

# 카드 시각화와 JSON 저장에 사용할 before/after 비교 report를 만듭니다.
# 각 sample에는 COCO id, 파일명, 정답 caption, zero-shot caption, fine-tuned caption을 넣습니다.
before_after_report = {
    "samples": [
        {
            "cocoid": int(sample["cocoid"]),
            "filename": str(sample["filename"]),
            "ground_truth": str(sample["caption"]),
            "references": [str(sample["caption"])],
            "zero_shot": zero_shot_predictions[index],
            "fine_tuned": fine_tuned_predictions[index],
        }
        for index, sample in enumerate(comparison_samples)
    ],
    "metrics": comparison_metrics,
}

# terminal/logger 출력으로도 metric 변화를 확인할 수 있게 요약합니다.
FINETUNE_LOGGER.info("Terminal before/after comparison")
FINETUNE_LOGGER.info(
    "Metrics | bleu=%.4f->%.4f (delta=%.4f) meteor=%.4f->%.4f (delta=%.4f) cider_d=%.4f->%.4f (delta=%.4f)",
    comparison_metrics["zero_shot"]["bleu"],
    comparison_metrics["fine_tuned"]["bleu"],
    comparison_metrics["delta"]["bleu"],
    comparison_metrics["zero_shot"]["meteor"],
    comparison_metrics["fine_tuned"]["meteor"],
    comparison_metrics["delta"]["meteor"],
    comparison_metrics["zero_shot"]["cider_d"],
    comparison_metrics["fine_tuned"]["cider_d"],
    comparison_metrics["delta"]["cider_d"],
)

# sample별 caption을 로그로 출력해 metric 숫자와 실제 문장 변화를 함께 볼 수 있게 합니다.
for index, record in enumerate(before_after_report["samples"], start=1):
    FINETUNE_LOGGER.info("Sample %s | filename=%s", index, record.get("filename", "n/a"))
    FINETUNE_LOGGER.info("GT: %s", record.get("ground_truth", "n/a"))
    FINETUNE_LOGGER.info("Zero-shot: %s", record.get("zero_shot", "n/a"))
    FINETUNE_LOGGER.info("Fine-tuned: %s", record.get("fine_tuned", "n/a"))


## 결과 저장

**핵심:** before/after report, 비교 이미지, 학습/평가 metric을 파일로 저장한다.

- `before_after.json`: caption 비교와 metric을 저장한다.
- `train_eval_metrics.json`: 설정, checkpoint 경로, 학습/평가 metric, log history를 저장한다.


In [ ]:
# 결과 저장

# sample별 before/after caption과 metric dict를 JSON으로 저장합니다.
save_json(config_ft.output_dir / "before_after.json", before_after_report)
FINETUNE_LOGGER.info("Saved before/after captions to %s", config_ft.output_dir / "before_after.json")

# notebook 밖에서도 확인할 수 있도록 before/after caption 비교 이미지를 파일로 저장합니다.
save_caption_comparison_figure(
    samples=comparison_samples,
    zero_shot_predictions=zero_shot_predictions,
    fine_tuned_predictions=fine_tuned_predictions,
    output_path=config_ft.output_dir / "before_after.png",
)
FINETUNE_LOGGER.info("Saved comparison figure to %s", config_ft.output_dir / "before_after.png")

# 학습 설정, checkpoint 경로, train/eval metric, Trainer log history를 함께 저장합니다.
train_eval_metrics = {
    "config": config_payload,
    "checkpoint_dir": str(checkpoint_dir),
    "used_existing_checkpoint": used_existing_checkpoint,
    "train_result": train_result_metrics,
    "eval_metrics": eval_metrics,
    "log_history": log_history,
}
save_json(config_ft.output_dir / "train_eval_metrics.json", train_eval_metrics)
FINETUNE_LOGGER.info("Saved train/eval metrics to %s", config_ft.output_dir / "train_eval_metrics.json")


## Before / After 비교 및 시각화

**핵심:** fine-tuning 전후 caption과 metric 변화를 같은 이미지 기준으로 비교한다.

- **해석 포인트:** 모든 caption이 좋아지는지보다 어떤 유형의 묘사가 개선/악화되는지 관찰한다.


In [ ]:
# Before / After 비교 및 시각화

# Section 4의 주요 결과를 result_ft에 모아 이후 분석이나 재시각화에서 바로 사용할 수 있게 합니다.
result_ft = {
    "config": config_ft,
    "device": device,
    "device_info": device_info_payload,
    "train_image_samples": train_image_samples,
    "val_image_samples": val_image_samples,
    "test_image_samples": test_image_samples,
    "comparison_samples": comparison_samples,
    "zero_shot_predictions": zero_shot_predictions,
    "fine_tuned_predictions": fine_tuned_predictions,
    "comparison_metrics": comparison_metrics,
}

# 같은 이미지에 대해 fine-tuning 전후 caption을 나란히 카드로 보여줍니다.
show_caption_comparison_cards(
    result_ft["comparison_samples"],
    prediction_sets={
        "Zero-shot": result_ft["zero_shot_predictions"],
        "Fine-tuned": result_ft["fine_tuned_predictions"],
    },
)

# test split metric을 before/after 막대그래프로 비교합니다.
_ = show_caption_metric_comparison(
    {
        "Zero-shot": result_ft["comparison_metrics"]["zero_shot"],
        "Fine-tuned": result_ft["comparison_metrics"]["fine_tuned"],
    },
    title="Before/after caption metrics",
)


## 결과 Drive 복사

**핵심:** Colab 환경의 `/content/` 경로에 저장된 최종 결과물(모델 checkpoint, json, png 파일 등)을 Google Drive 프로젝트 폴더로 백업한다.

In [ ]:
# 결과 Drive 복사

# Colab에서 Google Drive를 mount한 경우 결과 폴더를 Drive 프로젝트 아래로 복사하기 위해 사용합니다.
import shutil

# 앞쪽 Colab 환경 설정 셀에서 drive 객체가 만들어진 경우에만 복사를 시도합니다.
if 'drive' in globals():
    # DRIVE_PROJECT와 config_ft가 있어야 Drive 안의 대상 경로를 계산할 수 있습니다.
    if 'DRIVE_PROJECT' in globals() and 'config_ft' in globals():
        drive_target_dir = Path(DRIVE_PROJECT) / config_ft.output_dir
        print(f"결과를 Google Drive로 복사합니다:\n  Source: {config_ft.output_dir.absolute()}\n  Target: {drive_target_dir}")

        # output_dir 전체를 Drive로 복사합니다. 같은 폴더가 이미 있으면 파일을 덮어씁니다.
        shutil.copytree(config_ft.output_dir, drive_target_dir, dirs_exist_ok=True)
        print("\n✅ 복사가 성공적으로 완료되었습니다!")
    else:
        # 필요한 변수가 없다면 앞선 셀이 실행되지 않은 것이므로 안내만 출력합니다.
        print("DRIVE_PROJECT 또는 config_ft 변수를 찾을 수 없습니다. 이전 셀들을 정상적으로 실행했는지 확인해주세요.")
else:
    # 로컬 실행에서는 Drive mount가 없으므로 복사 단계를 건너뜁니다.
    print("Colab 환경이 아니므로 Drive 복사를 건너뜁니다.")
